# Final Assessment (Bài kiểm tra cuối khóa)

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

Notebook này giúp bạn **ôn tập & tự kiểm tra** toàn bộ khóa học qua 3 phần:
1. Trắc nghiệm khái niệm (tự chấm)
2. Bài tập thực hành (capstone)
3. Bảng tổng kết kiến thức


## Phần 1 — Câu hỏi ôn tập

Tự trả lời, sau đó mở ô đáp án bên dưới để đối chiếu.

1. Ba thành phần **bắt buộc** của một request tới `messages.create` là gì?
2. API của Claude có **tự nhớ** lịch sử hội thoại không? Vì sao?
3. Khi `stop_reason == "max_tokens"` nghĩa là gì?
4. Trong "LLM as a Judge", vì sao nên cho model viết `reasoning` **trước** `score`?
5. Kỹ thuật **prefill** (mồi `assistant`) còn dùng được trên Opus 4.8 không? Thay bằng gì?
6. Khi Claude trả về `stop_reason == "tool_use"`, **ai** thực thi công cụ?
7. Khác biệt chính giữa **RAG cổ điển** và **Agentic Search**?
8. MCP giải quyết vấn đề gì so với tự viết tool use?
9. Nêu **4 tiêu chí** để quyết định có nên xây Agent.
10. Trên model mới (Opus 4.6+), thay cho `temperature` ta dùng gì để điều chỉnh độ "suy nghĩ"?


<details>
<summary><b>👉 Bấm xem đáp án</b></summary>

1. `model`, `max_tokens`, `messages`.
2. **Không** — API là *stateless*; phải gửi lại toàn bộ lịch sử mỗi lần gọi.
3. Câu trả lời bị **cắt** vì chạm giới hạn `max_tokens` → tăng `max_tokens` hoặc dùng streaming.
4. Bắt model **suy nghĩ trước khi chấm** → điểm chính xác & nhất quán hơn (chain-of-thought).
5. **Không** còn dùng được (lỗi 400). Thay bằng **`output_config`** (structured output) hoặc ra lệnh thẳng trong prompt.
6. **Code của bạn** thực thi — Claude chỉ *yêu cầu*, không tự chạy công cụ.
7. RAG cổ điển: **tìm 1 lần** rồi trả lời. Agentic Search: Claude **tự tìm nhiều lần**, tinh chỉnh truy vấn (tool use).
8. MCP = **chuẩn mở**: viết tích hợp **một lần, dùng mọi nơi**, tái sử dụng; tránh viết lại tích hợp cho từng app.
9. **Phức tạp, Giá trị, Khả thi, Sửa được lỗi** (thiếu 1 → dùng workflow).
10. **`output_config={"effort": "low|medium|high|max"}`** (kèm adaptive thinking).
</details>


## Phần 2 — Bài tập Capstone 🏆

**Đề:** Xây một **trợ lý hỗ trợ khách hàng** kết hợp nhiều kỹ thuật đã học:
- Một **công cụ tra cứu đơn hàng** (tool use).
- **Agentic loop** để Claude tự gọi công cụ khi cần.
- System prompt **gán vai trò**.

Hãy đọc và chạy lời giải mẫu dưới đây, rồi thử **mở rộng** (thêm công cụ hủy đơn, đổi địa chỉ...).


In [ ]:
import anthropic
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

# "Database" đơn hàng giả lập
orders = {
    "DH001": {"san_pham": "Tai nghe Bluetooth", "trang_thai": "Đang giao", "ngay_du_kien": "25/06"},
    "DH002": {"san_pham": "Bàn phím cơ", "trang_thai": "Đã giao", "ngay_du_kien": "20/06"},
}

# 1. Công cụ tra cứu đơn hàng
tools = [{
    "name": "lookup_order",
    "description": "Tra cứu trạng thái đơn hàng theo mã đơn (vd 'DH001').",
    "input_schema": {
        "type": "object",
        "properties": {"order_id": {"type": "string", "description": "Mã đơn hàng"}},
        "required": ["order_id"],
    },
}]

def lookup_order(order_id):
    o = orders.get(order_id.upper())
    if not o:
        return f"Không tìm thấy đơn {order_id}."
    return f"Sản phẩm: {o['san_pham']}, Trạng thái: {o['trang_thai']}, Dự kiến: {o['ngay_du_kien']}"

# 2. Agentic loop + 3. Role
SYSTEM = "Bạn là trợ lý CSKH thân thiện, lịch sự. Khi khách hỏi về đơn hàng, hãy tra cứu rồi trả lời rõ ràng."

def support_agent(user_msg):
    messages = [{"role": "user", "content": user_msg}]
    while True:
        res = client.messages.create(model=MODEL, max_tokens=1024,
                                     system=SYSTEM, tools=tools, messages=messages)
        if res.stop_reason == "end_turn":
            return next(b.text for b in res.content if b.type == "text")
        messages.append({"role": "assistant", "content": res.content})
        results = []
        for b in res.content:
            if b.type == "tool_use":
                print(f"  🔧 tra cứu {b.input}")
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": lookup_order(b.input["order_id"])})
        messages.append({"role": "user", "content": results})

print(support_agent("Cho mình hỏi đơn DH001 bao giờ giao tới?"))

### 🎯 Thử thách mở rộng (tự làm)
1. Thêm công cụ `cancel_order(order_id)` để hủy đơn.
2. Dùng **structured output** cho công cụ trả về JSON thay vì chuỗi.
3. Viết một **eval** (bài Prompt Evaluation) để chấm điểm chất lượng câu trả lời của agent.
4. Thêm **RAG**: tra cứu thêm chính sách đổi/trả khi khách hỏi.


## Phần 3 — Bản đồ kiến thức toàn khóa 🗺️

| Chương | Khái niệm cốt lõi |
|---|---|
| **Accessing the API** | client, messages, roles, tokens, system prompt, stateless, streaming |
| **Prompt Evaluation** | dataset → run → grade → aggregate; LLM as a Judge |
| **Prompt Engineering** | rõ ràng, vai trò, few-shot, XML tags, chain-of-thought, chaining |
| **Tool Use** | định nghĩa tool, vòng lặp agentic, tool_result, is_error |
| **RAG & Agentic Search** | retrieve→augment→generate; search thành tool |
| **MCP** | chuẩn mở Host→Client→Server; viết 1 lần dùng mọi nơi |
| **Anthropic Apps** | Claude Code (coding agent), Computer Use (điều khiển GUI) |
| **Agents & Workflows** | workflow (chaining/routing/parallel) vs agent; 4 tiêu chí |

> ⚠️ **Lưu ý xuyên suốt (model mới Opus 4.8):** không dùng `temperature`/`top_p` (→ `output_config.effort`); không dùng prefill (→ `output_config.format`); dùng `thinking={"type":"adaptive"}` cho suy luận sâu.


---
## 🎓 Chúc mừng bạn đã hoàn thành khóa học!

Bạn đã đi từ lệnh gọi API đầu tiên đến xây dựng agent hoàn chỉnh. Bước tiếp theo gợi ý:
- Làm **thử thách mở rộng** ở trên.
- Đọc tài liệu chính thức: [platform.claude.com/docs](https://platform.claude.com/docs)
- Thử **Claude Code** cho dự án thật của bạn.
- Áp dụng **Prompt Evaluation** vào sản phẩm để cải tiến liên tục.
